# Port Data Catalog Updater - Version 2

In [73]:
import pandas as pd

# ======================================================
# CONFIG — CHANGE THESE EACH YEAR
# ======================================================
REPORTING_YEAR = 2022

EXCEL_FILE = f"PortPerformance{REPORTING_YEAR}.xlsx"
HISTORICAL_FILE = "Port_Data_20251112.csv"
OUTPUT_FILE = f"Annual_Port_Statistics_{REPORTING_YEAR}_UPDATE.csv"
#PERCENT_FILE = f"PortPerformance{REPORTING_YEAR-1}.xlsx"
# ======================================================
# LOAD WORKBOOK + HISTORICAL DATA
# ======================================================
xls = pd.ExcelFile(EXCEL_FILE)
#xls_minus1 = pd.ExcelFile(PERCENT_FILE)
df_hist = pd.read_csv(HISTORICAL_FILE)
df_hist['Port ID'] = df_hist['Port ID'].str.replace(",","").astype(int)

df_hist['Percent Change'] = df_hist['Percent Change'].str.replace(",","").astype(float) 
df_hist['Percent Change'] = df_hist['Percent Change'].round(1) 
df_hist_year = df_hist[df_hist["Reporting Year"] == REPORTING_YEAR]
df_hist_year_minus_1 = df_hist[df_hist["Reporting Year"] == REPORTING_YEAR - 1]
#df_hist_year_minus_1['Port ID'] = df_hist_year_minus_1['Port ID'].str.replace(",","").astype(int)
df_hist_year_minus_1['Volume'] = df_hist_year_minus_1['Volume'].str.replace(",","").astype(float)
records = []

if REPORTING_YEAR == 2022:
    portXrefs = pd.read_csv("Port_Xrefs.csv")
    portXrefs=portXrefs.set_index("Known_Port_Name", drop=True)['Best_Match'].to_dict()
    df_container = pd.read_excel("WCSC 2022 Container traffic.xlsx", header=[0, 1, 2])
else:
    df_container = pd.read_excel("2023 Annual TEUs from US Army.xlsx", header=[0, 1, 2])
    portXrefs = pd.read_csv("Port_Xrefs.csv")
    portXrefs=portXrefs.set_index("Known_Port_Name", drop=True)['Best_Match'].to_dict()
    df_hist_year_minus_1 = pd.read_csv(f"Port_Performance_{REPORTING_YEAR-1}_for_Socrata_Joe.csv")



/tmp/ipykernel_113479/3344805943.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hist_year_minus_1['Volume'] = df_hist_year_minus_1['Volume'].str.replace(",","").astype(float)


## Helper Functions

### Get State

In [74]:
tmp = xls.parse("Ports by ICST")
# df['State'] = df['PORT_NAME'].str.split(', ').str[-1]
# df.head()
# state_abbrev_to_name = {
#     'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
#     'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
#     'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
#     'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
#     'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
#     'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
#     'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
#     'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
#     'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
#     'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
#     'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
#     'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
#     'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia',
#     'PR': 'Puerto Rico', 'VI': 'Virgin Islands', 'GU': 'Guam', 'AS': 'American Samoa'
# }

# Usage
#df['State_Full'] = df['State'].map(state_abbrev_to_name)
tmp.head()


,PORT,PORT_NAME,CATEGORY,INBOUND,OUTBOUND,TOTAL
0,77,"Searsport, ME",Dry Bulk,1,0,1
1,77,"Searsport, ME",Dry Bulk Barge,2,2,4
2,77,"Searsport, ME",Other Freight,9,5,14
3,77,"Searsport, ME",Other Freight Barge,22,14,36
4,78,"Portsmouth, NH",Dry Bulk,23,11,34


### Get Station List form Historical Data

In [75]:
ports_hist = df_hist[['Port_Name','Port ID','State', 'Region']].drop_duplicates()
#ports_hist['Port ID'] = ports_hist['Port ID'].str.replace(",","")
ports_hist.dropna(inplace=True)
ports_hist.to_csv("ports_hist.csv", index=False)



## Total Tonage

In [76]:
df = xls.parse("Top Ports")
print("DF ORIG SHAPE",df.shape)
## Get Port ID from the POrts by ICST sheet
tmp = xls.parse("Ports by ICST")
tmp = tmp[["PORT","PORT_NAME"]].drop_duplicates()
df=pd.merge(df,tmp,how="left",left_on="PORT NAME",right_on="PORT_NAME")
df.drop(columns=["PORT_NAME"],inplace=True)
print("DF NEW SHAPE",df.shape)
# df['PORT'] = df['PORT'].astype(int) 

mapr = {'GRAND TOTAL':'TOTAL','FOREIGN TOTAL':'FOREIGN','IMPORTS':'IMPORTS','EXPORTS':'EXPORTS','DOMESTIC':'DOMESTIC'}
all_records = []
for idx,row in ports_hist.iterrows():
  #  print(row)
    port_id = row['Port ID']
    port_name = row['Port_Name'].strip()

    hld=df.loc[(df['PORT'] == port_id) | (df['PORT NAME'].str.strip() == port_name) ]
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOTAL TONNAGE')]
    
    if hld.shape[0]==0:
        print("No data for port ID:", port_id, row['Port_Name'])
        continue
    for tt in ['GRAND TOTAL','FOREIGN TOTAL','IMPORTS','EXPORTS','DOMESTIC']:
        tp = mapr[tt]
        vol_minus_1 = hst[hst['Trade Type'] == tp]['Volume'].values[0] if not hst.empty else None              
       
        if vol_minus_1:
            percent_change = ((hld[tt].values[0] - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
         #   print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tp}. Cannot calculate percent change.")




        tmp = {
          'Cargo Type': 'TOTAL TONNAGE', 
          'Port ID': int(port_id), 
          'Port_Name': row['Port_Name'], 
          'Region': row['Region'],
          'Reporting Year': REPORTING_YEAR,
          'State': row['State'], 
          'Trade Type':tp, 
          'Units':'Short Tons', 
          'Port Ranking': hld['RANK'].values[0],
          'Percent Change':percent_change,
          'Volume': hld[tt].values[0]
        }
       
        all_records.append(tmp)

totalTonnage = pd.DataFrame(all_records)
print("Total Tonnage shape:",totalTonnage.shape)


        

DF ORIG SHAPE (299, 8)
DF NEW SHAPE (299, 9)
No data for port ID: 2338 Cincinnati-Northern KY, Ports of
No data for port ID: 2083 Gulfport, MS
No data for port ID: 2348 Huntington-Tristate, KY, OH, WV
No data for port ID: 2367 St. Louis, MO and IL
Total Tonnage shape: (240, 11)


### Total Check

#### Internal Checks

In [77]:
nbad=0
ngood=0
ntotal=0

for pid in totalTonnage['Port ID'].unique():
    row = totalTonnage[totalTonnage['Port ID'] == pid]
    hld={}
    bad=False
    ntotal+=1
    for tt in row['Trade Type'].unique():
        hld[tt] = row[row['Trade Type'] == tt]['Volume'].values[0]
    frgn = hld['FOREIGN']
    dmst = hld['DOMESTIC']
    tot = hld['TOTAL']
    impt = hld['IMPORTS']
    expo = hld['EXPORTS']
    if frgn != impt + expo:
        print(f"Mismatch for Port ID {pid}: FOREIGN {frgn} vs IMPORTS+EXPORTS {impt+expo} IMPT {impt} EXPO {expo}")
        bad=True
    if tot != dmst + frgn:
        print(f"Mismatch for Port ID {pid}: TOTAL {tot} vs DOMESTIC+FOREIGN {dmst+frgn} DMST {dmst} FRGN {frgn}")
        bad=True

    if bad:
        nbad+=1
    else:
        ngood+=1

print(f"Checked {ntotal} ports with {nbad} mismatches and {ngood} good records.")

# Get value counts for the 'Trade Type' column
counts = totalTonnage['Trade Type'].value_counts()

# Check if all counts are equal
all_equal = counts.nunique() == 1
print(f"All types have equal counts: {all_equal}")

# See which types have different counts
print(f"\nMin count: {counts.min()}")
print(f"Max count: {counts.max()}")
print(f"Difference: {counts.max() - counts.min()}")
#    print(f"Port ID {pid}, Total Volume from Trade Types: {tot}, Total Volume from TOTAL row: {row[row['Trade Type'] == 'TOTAL']['Volume'].values[0]}")

Checked 48 ports with 0 mismatches and 48 good records.
All types have equal counts: True

Min count: 48
Max count: 48
Difference: 0


## Dry Bulk

In [78]:

df = xls.parse("Dry Bulk")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("DRY BULK TONNAGE COLUMNS:", df.columns.tolist())
df['PORT'] = df['PORT'].astype('Int64')
## Get Port ID from the POrts by ICST sheet
# tmp = xls.parse("Ports by ICST")
# tmp = tmp[["PORT","PORT_NAME"]].drop_duplicates()
# df=pd.merge(df,tmp,how="left",left_on="PORT NAME",right_on="PORT_NAME")
# df.drop(columns=["PORT_NAME"],inplace=True)
# print("DF NEW SHAPE",df.shape)

all_records = []
for idx,row in ports_hist.iterrows():
  #  print(row)
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()
    hld=df.loc[(df['PORT'] == port_id) | (df['PORT NAME'].str.strip() == port_name)]
    if hld.shape[0]==0:
        print("No data for port ID:",port_id,row['Port_Name'])
        continue
    
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'DRY BULK')]

    for tt in ['TOTAL','FOREIGN','IMPORTS','EXPORTS','DOMESTIC']:
        volume=int(hld[tt].values[0]) if pd.notna(hld[tt].values[0]) else 0
        vol_minus_1 = hst[hst['Trade Type'] == tt]['Volume'].values[0] if not hst.empty else None              
       
        if vol_minus_1:
            percent_change = ((hld[tt].values[0] - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")




        tmp = {
          'Cargo Type': 'DRY BULK', 
          'Port ID': port_id, 
          'Port_Name': row['Port_Name'], 
          'Region': row['Region'],
          'Reporting Year': REPORTING_YEAR,
          'State': row['State'], 
          'Trade Type':tt, 
          'Units':'Short Tons', 
          'Port Ranking': hld.index.values[0]+1,
          'Percent Change':percent_change,
          'Volume': volume
        }
       
        all_records.append(tmp)

dryBulk = pd.DataFrame(all_records)
print("Dry Bulk shape:",dryBulk.shape)

        

DRY BULK TONNAGE COLUMNS: ['PORT', 'RANK', 'PORT NAME', 'TOTAL', 'DOMESTIC', 'EXPORTS', 'IMPORTS', 'FOREIGN', 'TOTAL.1']
No historical volume for port ID Alaska, AK Port of 4820, trade type EXPORTS. Cannot calculate percent change.
No data for port ID: 2338 Cincinnati-Northern KY, Ports of
No data for port ID: 2083 Gulfport, MS
No historical volume for port ID Honolulu, O'ahu, HI 4421, trade type EXPORTS. Cannot calculate percent change.
No data for port ID: 2348 Huntington-Tristate, KY, OH, WV
No historical volume for port ID Northern Indiana, IN 3743, trade type EXPORTS. Cannot calculate percent change.
No historical volume for port ID Port Freeport, TX 2408, trade type EXPORTS. Cannot calculate percent change.
No historical volume for port ID Port of Charleston, SC 775, trade type EXPORTS. Cannot calculate percent change.
No data for port ID: 2367 St. Louis, MO and IL
No historical volume for port ID Two Harbors, MN 3926, trade type IMPORTS. Cannot calculate percent change.
Dry Bulk

### Dry Bulk Check

#### Internal Check

In [79]:
nbad=0
ngood=0
ntotal=0

for pid in dryBulk['Port ID'].unique():
    row = dryBulk[dryBulk['Port ID'] == pid]
    hld={}
    bad=False
    ntotal+=1
    for tt in row['Trade Type'].unique():
        hld[tt] = row[row['Trade Type'] == tt]['Volume'].values[0]
    frgn = hld['FOREIGN']
    dmst = hld['DOMESTIC']
    tot = hld['TOTAL']
    impt = hld['IMPORTS']
    expo = hld['EXPORTS']
    if frgn != impt + expo:
        print(f"Mismatch for Port ID {pid}: FOREIGN {frgn} vs IMPORTS+EXPORTS {impt+expo} IMPT {impt} EXPO {expo}")
        bad=True
    if tot != dmst + frgn:
        print(f"Mismatch for Port ID {pid}: TOTAL {tot} vs DOMESTIC+FOREIGN {dmst+frgn} DMST {dmst} FRGN {frgn}")
        bad=True

    if bad:
        nbad+=1
    else:
        ngood+=1

print(f"Checked {ntotal} ports with {nbad} mismatches and {ngood} good records.")

# Get value counts for the 'Trade Type' column
counts = dryBulk['Trade Type'].value_counts()

# Check if all counts are equal
all_equal = counts.nunique() == 1
print(f"All types have equal counts: {all_equal}")

# See which types have different counts
print(f"\nMin count: {counts.min()}")
print(f"Max count: {counts.max()}")
print(f"Difference: {counts.max() - counts.min()}")
#    print(f"Port ID {pid}, Total Volume from Trade Types: {tot}, Total Volume from TOTAL row: {row[row['Trade Type'] == 'TOTAL']['Volume'].values[0]}")

Checked 48 ports with 0 mismatches and 48 good records.
All types have equal counts: True

Min count: 48
Max count: 48
Difference: 0


## Vessel Calls

In [80]:
df = xls.parse("Ports by ICST")

all_records = []

for idx,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()
 #   tmp = df.loc[(df['PORT'] == port_id) ]
 #   print("Processing Port ID:", port_id, row['Port_Name'],tmp.shape)
    if df.loc[(df['PORT'] == port_id) | (df['PORT_NAME'].str.strip() == port_name)].shape[0]==0:
        print("No data for port ID:",port_id,row['Port_Name'])
        continue
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'VESSEL CALLS')]
    
    nmiss=0
    for tt in ["Container","Other Freight Barge","Dry Bulk","Dry Bulk Barge","Other Freight"]:

        tmp = df.loc[((df['PORT'] == port_id) | (df['PORT_NAME'].str.strip() == port_name)) & (df['CATEGORY'].str.strip() == tt)]
        vol_minus_1 = hst[hst['Trade Type'] == tt]['Volume'].values[0] if not hst.empty else None              
       
        
        if tmp.shape[0] > 0:
            vol_in = tmp['INBOUND'].values[0]
            vol_out = tmp['OUTBOUND'].values[0]
            vol_mean = (vol_in + vol_out) / 2
            
        else:
            print("MIS ",port_id,tt,tmp.shape)
            vol_mean=0
            nmiss+=1

        if vol_minus_1 and vol_minus_1 > 0:
            val = tmp.loc[tmp['CATEGORY'].str.strip() == tt, 'INBOUND'].values[0] if not tmp.empty else 0   
            percent_change = ((vol_mean - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")

        hld = {
            'Cargo Type': 'VESSEL CALLS', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Vessel Calls', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': vol_mean
            }
        all_records.append(hld)
    if nmiss == 5:
        print("No data for Port ID:", port_id, row['Port_Name'])
vesselCalls = pd.DataFrame(all_records)
print("Vessel Calls shape:",vesselCalls.shape)



MIS  2393 Container (0, 6)
No historical volume for port ID Beaumont, TX 2393, trade type Container. Cannot calculate percent change.
No data for port ID: 2338 Cincinnati-Northern KY, Ports of
MIS  2436 Container (0, 6)
No historical volume for port ID Corpus Christi, TX 2436, trade type Container. Cannot calculate percent change.
MIS  3924 Container (0, 6)
No historical volume for port ID Duluth-Superior, MN and WI 3924, trade type Container. Cannot calculate percent change.
MIS  2252 Container (0, 6)
No historical volume for port ID Greater Baton Rouge, LA Port of 2252, trade type Container. Cannot calculate percent change.
No data for port ID: 2083 Gulfport, MS
No data for port ID: 2348 Huntington-Tristate, KY, OH, WV
MIS  4626 Container (0, 6)
No historical volume for port ID Kalama, WA Port of 4626, trade type Container. Cannot calculate percent change.
MIS  2248 Container (0, 6)
No historical volume for port ID Lake Charles Harbor District, LA 2248, trade type Container. Cannot c

## Top 5 Commodities

In [81]:
## Fix this to get commodities out of 2021 port performance list


df = xls.parse("Ports by Commodity")

all_records = []

for _,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()

    hld=df.loc[(df['PORT'] == port_id) | (df['Port Name'].str.strip() == port_name)]
    total = hld['TOTAL'].sum()
    hld=hld.sort_values(by="TOTAL", ascending=False).reset_index(drop=True).iloc[:5]
    for idx,row2 in hld.iterrows():
        tt=row2['Commodity Name'].strip() 
        volume=row2['TOTAL']
        hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == tt)]
        vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
        if vol_minus_1 and vol_minus_1 > 0:
            val = row2['TOTAL']   
            percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")




        tmp = {
            'Cargo Type': 'TOP 5 COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': volume
            }
        if volume> 0:
           all_records.append(tmp)
## Add Total
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == 'TOTAL')]
    vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
    if vol_minus_1 and vol_minus_1 > 0:
        val = total   
        percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
        percent_change = round(percent_change, 1)
    else:
        percent_change = 0
        print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")

    tmp = {
            'Cargo Type': 'TOP 5 COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':'TOTAL', 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':None,
            'Volume': total
            }
    if total > 0:
       all_records.append(tmp)

top5Com = pd.DataFrame(all_records)

print("Top 5 Commodities shape:",top5Com.shape)


    
    

No historical volume for port ID Alaska, AK Port of 4820, trade type Gasoline. Cannot calculate percent change.
No historical volume for port ID Baltimore, MD 700, trade type Sand & Gravel. Cannot calculate percent change.
No historical volume for port ID Boston, MA 149, trade type Cement & Concrete. Cannot calculate percent change.
No historical volume for port ID Boston, MA 149, trade type Pulp & Waste Paper. Cannot calculate percent change.
No historical volume for port ID Cincinnati-Northern KY, Ports of 2338, trade type Pulp & Waste Paper. Cannot calculate percent change.
No historical volume for port ID Tacoma, WA 4719, trade type Iron & Steel Scrap. Cannot calculate percent change.
No historical volume for port ID Greater Baton Rouge, LA Port of 2252, trade type Petroleum Coke. Cannot calculate percent change.
No historical volume for port ID Gulfport, MS 2083, trade type Petroleum Coke. Cannot calculate percent change.
No historical volume for port ID Honolulu, O'ahu, HI 4421, 

## Top 5 Farm/Agriculture Commodities

In [82]:
top5Ag_hist = df_hist.loc[df_hist['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES']
ag_commodities_hist = top5Ag_hist['Trade Type'].unique().tolist()
df = xls.parse("Ports by Commodity")
all_records = []

for _,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    port_name = row['Port_Name'].strip()
 #   hld=top5Ag.loc[top5Ag['PORT'] == port_id]
    total = df.loc[((df['PORT'] == port_id) | (df['Port Name'].str.strip() == port_name)) & (df['Commodity Group'] >= 6000) & (df['Commodity Group'] < 7000),'TOTAL'].sum()  
    hld = df.loc[((df['PORT'] == port_id) | (df['Port Name'].str.strip() == port_name)) & (df['Commodity Group'] >= 6000) & (df['Commodity Group'] < 7000)]
    hld=hld.sort_values(by="TOTAL", ascending=False).reset_index(drop=True).iloc[:5]
    for idx,row2 in hld.iterrows():
        tt=row2['Commodity Name'].strip() 
        volume=row2['TOTAL']
        hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == tt)]
        vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
        if vol_minus_1 and vol_minus_1 > 0:
            val = row2['TOTAL']   
            percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")


        tmp = {
            'Cargo Type': 'TOP 5 FOOD/FARM COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': volume
            }
        if volume> 0:
           all_records.append(tmp)
## Add Total
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == 'TOTAL')]
    vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
    if vol_minus_1 and vol_minus_1 > 0:
        val = total   
        percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
        percent_change = round(percent_change, 1)
    else:
        percent_change = 0
        print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")



    tmp = {
            'Cargo Type': 'TOP 5 FOOD/FARM COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':'TOTAL', 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': total
            }
    if total > 0:
       all_records.append(tmp)

top5Ag = pd.DataFrame(all_records)

print("Top 5 Farm/Food Commodities shape:",top5Ag.shape)




No historical volume for port ID Beaumont, TX 2393, trade type Fruit & Nuts NEC. Cannot calculate percent change.
No historical volume for port ID Beaumont, TX 2393, trade type Coffee. Cannot calculate percent change.
No historical volume for port ID Beaumont, TX 2393, trade type Food Products NEC. Cannot calculate percent change.
No historical volume for port ID Beaumont, TX 2393, trade type Grain Mill Products. Cannot calculate percent change.
No historical volume for port ID Boston, MA 149, trade type Vegetables & Prod.. Cannot calculate percent change.
No historical volume for port ID Cincinnati-Northern KY, Ports of 2338, trade type Vegetables & Prod.. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Grain Mill Products. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Sugar. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga

## Container Activity

In [83]:
df = df_container.copy()
a=df.columns.tolist()
colsNew = []
for b in a:
    final=""

    for c in b:
        if "Unnamed:" in c:
            continue
        else:
            final += c.strip() + " "
    colsNew.append(final.strip())
    
df.columns = colsNew


df['IMPORTS'] = df['FOREIGN InBound Loaded'] 
df['EXPORTS'] = df['FOREIGN OutBound Loaded'] 
# df['EXPORTS'] = df['FOREIGN OutBound Loaded'] 
df['EMPTY'] = df['DOMESTIC InBound Empty'] + df['DOMESTIC OutBound Empty'] 
#df['TOTAL'] = df['Grand Total Loaded']
df['TOTAL'] = df['IMPORTS'] + df['EXPORTS']

df_contain = df[[ 'PORT NAME', 'STATE','TOTAL', 'IMPORTS','EXPORTS', 'EMPTY','FOREIGN InBound Loaded','FOREIGN OutBound Loaded','DOMESTIC InBound Loaded','DOMESTIC OutBound Loaded']].copy()
if REPORTING_YEAR == 2023:
    df_contain['PORT CODE'] = df['PORT CODE'].astype(int)     
else:
    df_contain['PORT CODE'] = None


In [84]:
df_contain.columns

Index(['PORT NAME', 'STATE', 'TOTAL', 'IMPORTS', 'EXPORTS', 'EMPTY',
       'FOREIGN InBound Loaded', 'FOREIGN OutBound Loaded',
       'DOMESTIC InBound Loaded', 'DOMESTIC OutBound Loaded', 'PORT CODE'],
      dtype='object')

In [85]:
ranks=df_contain[['PORT CODE','PORT NAME','TOTAL']].sort_values(by='TOTAL', ascending=False).reset_index(drop=True)

all_records = []
for _,row in ports_hist.iterrows():
    pid = row['Port ID']
    pname = row['Port_Name'].strip()
    xref = portXrefs[pname]
    hld = df_contain.loc[(df_contain['PORT CODE'] == pid) | (df_contain['PORT NAME'].str.strip() == xref)  ]
    rank = ranks.loc[(ranks['PORT CODE'] == pid) | (ranks['PORT NAME'].str.strip() == xref)].index.values[0] + 1 if not ranks.loc[(ranks['PORT CODE'] == pid) | (ranks['PORT NAME'].str.strip() == xref)].empty else None
    if hld.shape[0] > 0:
         for tt in ['TOTAL','IMPORTS','EXPORTS','EMPTY','FOREIGN InBound Loaded', 'FOREIGN OutBound Loaded',
       'DOMESTIC InBound Loaded', 'DOMESTIC OutBound Loaded']:
            volume = hld[tt].values[0] if not hld.empty else 0
            volume = round(volume, 1) 
            hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == pid) & (df_hist_year_minus_1['Cargo Type'] == 'CONTAINER') & (df_hist_year_minus_1['Trade Type'] == tt)]
            vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
            vol_minus_1 = round(vol_minus_1, 1)
            if vol_minus_1 and vol_minus_1 > 0:  
                percent_change = ((volume - vol_minus_1) / vol_minus_1) * 100
                percent_change = round(percent_change, 1)
            else:
                percent_change = 0
             #   print(f"No historical volume for port ID {row['Port_Name']} {pid}, trade type {tt}. Cannot calculate percent change.")
            tmp = {
                    'Cargo Type': 'CONTAINER', 
                    'Port ID': pid, 
                    'Port_Name': pname, 
                    'Region': row['Region'],
                    'Reporting Year': REPORTING_YEAR,
                    'State': row['State'], 
                    'Trade Type':tt, 
                    'Units':'Container TEUs', 
                    'Port Ranking': rank,
                    'Percent Change':percent_change,
                    'Volume': volume
            }
            all_records.append(tmp)
                
    else:
        print("No data for Port ID:", pid, row['Port_Name'])

container = pd.DataFrame(all_records)
print("Container shape:",container.shape)



No data for Port ID: 2338 Cincinnati-Northern KY, Ports of
No data for Port ID: 2348 Huntington-Tristate, KY, OH, WV
No data for Port ID: 4626 Kalama, WA Port of
No data for Port ID: 2306 Mid-America Port Commission
No data for Port ID: 2351 New Bourbon Port, MO
No data for Port ID: 3743 Northern Indiana, IN
No data for Port ID: 2358 Pittsburgh, PA Port of
No data for Port ID: 2363 Southern Indiana Maritime District, IN
No data for Port ID: 2367 St. Louis, MO and IL
No data for Port ID: 3204 Toledo-Lucas County Port, OH
No data for Port ID: 3926 Two Harbors, MN
Container shape: (328, 11)


#### Internal Check

In [86]:
good=0
bad=0
nochk=0
for _,row in container.iterrows():
    pid = row['Port ID']
    pname = row['Port_Name'].strip()
    tt = row['Trade Type']
    vol = round(row['Volume'],1)
    pct = round(row['Percent Change'],1)
    tmp = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == pid) & (df_hist_year_minus_1['Cargo Type'] == 'CONTAINER') & (df_hist_year_minus_1['Trade Type'] == tt)]
    if not tmp.empty:
        voln1 = tmp['Volume'].values[0]
        voln1 = round(voln1,1)
        pp = ((vol - voln1) / voln1) * 100 if voln1 > 0 else 0
        pp = round(pp, 1)
        if abs(pp - pct) > 0.1:
            print(f"Discrepancy for Port ID {pid}, Trade Type {tt}: Calculated Percent Change {pp} vs Recorded Percent Change {pct}. Volume: {vol}, Volume-1: {voln1}")
            bad += 1
        else:
            good += 1
    else:
        nochk += 1

print(f"Checked {good + bad + nochk} records with {good} matches, {bad} discrepancies, and {nochk} unchecked."  )

Checked 328 records with 156 matches, 0 discrepancies, and 172 unchecked.


In [87]:
totals={}
for _,row in container.iterrows():
    pid = row['Port_Name']
    if pid not in totals:
        totals[pid] = {'Org':0,'New':0,'EMPTY':0}
    if row['Trade Type'] == 'TOTAL':
        totals[pid]['Org'] = row['Volume']
    elif row['Trade Type'] == 'IMPORTS' or row['Trade Type'] == 'EXPORTS':
        totals[pid]['New'] += row['Volume']
        if row['Trade Type'] == 'EMPTY':
            totals[pid]['EMPTY'] += row['Volume']
        
        

In [88]:
ngood=0
nbad=0

for port,dct in totals.items():
    org = round(dct['Org'])
    new = round(dct['New'])
    empty = round(dct['EMPTY'])
    if org != new:
        print(f"Port {port}: diff {new - org} Emp {empty}  Org {org} New {new}")   
        nbad+=1
    else:
        ngood+=1
        # if empty == 0:
        #     print(f"EMPTY NOT: Port {port}: Match with EMPTY {empty} Org {org} New {new}")

print(f"Checked {ngood + nbad} ports with {nbad} mismatches and {ngood} good records.") 

Port Houston Port Authority, TX: diff -1 Emp 0  Org 3251753 New 3251752
Checked 41 ports with 1 mismatches and 40 good records.


## Combine All Records  

In [89]:
allData = pd.concat([totalTonnage, dryBulk,vesselCalls, top5Com, top5Ag, container], ignore_index=True)
## In 2022, the Boston, MA port ID was changed from 149 to 90, and indicates a different port 
### as such, the Boston Port Id is changed to 90 as the processing code assigned 149 automatically to Boston  
allData.loc[allData['Port_Name'] == 'Boston, MA','Port ID'] = 90

allData.to_csv(f"Port_Performance_{REPORTING_YEAR}_for_Socrata_Joe.csv", index=False)   


### Extra

In [1]:
import pandas as pd
HISTORICAL_FILE = "Port_Data_20251112.csv"

df_hist = pd.read_csv(HISTORICAL_FILE)
df_hist_2021 = df_hist[df_hist["Reporting Year"] == 2021]

In [2]:
df_2022 = pd.read_csv(f"Port_Performance_2022_for_Socrata_Joe.csv")   
df_2023 = pd.read_csv(f"Port_Performance_2023_for_Socrata_Joe.csv")   


In [5]:
df_2022.shape[0],df_2023.shape[0]

(1606, 1591)

In [3]:
df_2022_2023 = pd.concat([df_2022, df_2023], ignore_index=True)
df_2022_2023.shape

(3197, 11)

In [7]:
df_hist['Cargo Type'].value_counts()

Cargo Type
TOTAL TONNAGE                  1445
DRY BULK                       1440
CONTAINER                       700
TOP 5 COMMODITIES               583
TOP 5 FOOD/FARM COMMODITIES     531
VESSEL CALLS                    490
Name: count, dtype: int64

In [9]:
df_hist_2021['Cargo Type'].value_counts()

Cargo Type
TOP 5 COMMODITIES              286
TOP 5 FOOD/FARM COMMODITIES    259
TOTAL TONNAGE                  240
DRY BULK                       240
VESSEL CALLS                   240
CONTAINER                      156
Name: count, dtype: int64

In [8]:
df_2022_2023['Cargo Type'].value_counts()

Cargo Type
CONTAINER                      648
TOP 5 COMMODITIES              570
TOP 5 FOOD/FARM COMMODITIES    539
TOTAL TONNAGE                  480
DRY BULK                       480
VESSEL CALLS                   480
Name: count, dtype: int64

In [11]:
df_all = pd.concat([df_hist, df_2022_2023], ignore_index=True)

In [14]:
print(df_all.groupby(['Cargo Type','Reporting Year'])['Volume'].count())

Cargo Type                   Reporting Year
CONTAINER                    2016              104
                             2017              108
                             2018              108
                             2019              108
                             2020              116
                             2021              156
                             2022              328
                             2023              320
DRY BULK                     2016              220
                             2017              235
                             2018              235
                             2019              225
                             2020              233
                             2021              221
                             2022              240
                             2023              240
TOP 5 COMMODITIES            2020              297
                             2021              286
                             2022     

In [4]:
df_2022_2023.to_csv("Port_Performance_2022_2023_for_Socrata_Joe.csv", index=False)          

In [93]:
ah = df_hist_2021['Cargo Type'].value_counts().sort_index().to_dict()
a2 = df_2022['Cargo Type'].value_counts().sort_index().to_dict()
a3 = df_2023['Cargo Type'].value_counts().sort_index().to_dict()

In [98]:
for tt in ah.keys():
    c1 = ah[tt]
    c2 = a2[tt]
    c3 = a3[tt] 
    print(f"Cargo Type: {tt:>30s}, 2021 Count: {c1:<5d}, 2022 Count: {c2:<5d}, 2023 Count: {c3:<5d}")


Cargo Type:                      CONTAINER, 2021 Count: 156  , 2022 Count: 328  , 2023 Count: 320  
Cargo Type:                       DRY BULK, 2021 Count: 240  , 2022 Count: 240  , 2023 Count: 240  
Cargo Type:              TOP 5 COMMODITIES, 2021 Count: 286  , 2022 Count: 284  , 2023 Count: 286  
Cargo Type:    TOP 5 FOOD/FARM COMMODITIES, 2021 Count: 259  , 2022 Count: 274  , 2023 Count: 265  
Cargo Type:                  TOTAL TONNAGE, 2021 Count: 240  , 2022 Count: 240  , 2023 Count: 240  
Cargo Type:                   VESSEL CALLS, 2021 Count: 240  , 2022 Count: 240  , 2023 Count: 240  


In [101]:
b1=df_hist_2021['Volume'].str.replace(",","").astype(float).value_counts().sort_index()
b2=df_2022['Volume'].value_counts().sort_index()
b3=df_2023['Volume'].value_counts().sort_index()


In [112]:
m1=df_hist_2021['Percent Change'].max()
m2=df_2022['Percent Change'].max()
m3=df_2023['Percent Change'].max()
m1,m2,m3

('99.555701073', 411800.0, 183619.0)

In [119]:
p2 = df_2022[df_2022['Percent Change'] > 200]

In [146]:
for _,row in p2.iterrows():
    name = row['Port_Name'].strip()
    pid = row['Port ID']
    cargo = row['Cargo Type'].strip()
    tt = row['Trade Type'].strip()
    vol = row['Volume']
 #   print(name,pid,cargo,tt)
    tmp = df_hist_2021.loc[(df_hist_2021['Port ID'].str.replace(",", "") == str(pid)) & (df_hist_2021['Cargo Type'] == cargo) & (df_hist_2021['Trade Type'] == tt)]
 #   tmp = df_hist_2021.loc[(df_hist_2021['Port ID'].str.replace(",", "") == str(pid)) ]
    if not tmp.empty:
        voln1 = tmp['Volume'].str.replace(",", "").astype(float).values[0]
        pct = row['Percent Change']
        print(f"Port {name:<35s} {pid:4d} {cargo:<30s}   {tt:>20s}   {voln1:8.1f}  {vol:8.1f}  {pct:8.1f}") 

Port Port of Charleston, SC               775 DRY BULK                                     DOMESTIC     8114.0   43686.0     438.4
Port PortMiami, FL                       1992 DRY BULK                                      IMPORTS       21.0    3204.0   15157.1
Port Texas City, TX                      2428 DRY BULK                                      IMPORTS        1.0    4119.0  411800.0
Port Honolulu, O'ahu, HI                 4421 VESSEL CALLS                      Other Freight Barge       63.5     287.5     352.8
Port New Bourbon Port, MO                2351 VESSEL CALLS                      Other Freight Barge        0.5      11.0    2100.0
Port Honolulu, O'ahu, HI                 4421 TOP 5 COMMODITIES                  Manufac. Prod. NEC   2762606.0  10249219.0     271.0
Port Kalama, WA Port of                  4626 TOP 5 COMMODITIES                      Sorghum Grains   145505.0  1219553.0     738.2
Port Port Arthur, TX                     2416 TOP 5 COMMODITIES                

In [153]:
df_hist.loc[df_hist['Percent Change'] > 200,'Percent Change'].value_counts().sort_index(ascending=False)

Percent Change
879580.6    1
310512.0    1
201945.8    1
150589.6    1
51326.6     1
           ..
216.6       1
216.2       2
203.0       1
202.8       2
202.4       1
Name: count, Length: 82, dtype: int64

In [149]:
df_hist['Reporting Year'].value_counts()

Reporting Year
2020    1465
2021    1421
2019     588
2017     583
2018     583
2016     549
Name: count, dtype: int64